# ASE Interface

*Lucas Visscher, Vrije Universiteit Amsterdam, 2025*

Since release DIRAC25 we provide a Python interface to the Atomistic Simulation Environment that can be used to automate calculations.

A prerequisite to this interface is to follow the installation instructions in the `ase_dirac` folder. As a minimum you need to do:

1. [install the ASE environment](https://wiki.fysik.dtu.dk/ase/install.html) (if not already installed)
2. add DIRAC's build directory to your PATH (as the scripts need the `pam` command)
3. `cd ase_dirac; pip install --user -e .; cd ..` (to install the ase_dirac interface)

After this step you may automate DIRAC calculations. The next code cell checks whether the interface is installed correctly. If it gives an error, you should check whether there was an issue with any of the three required steps above.

In [ ]:
from ase import Atoms
from ase.io import read
from ase.build import molecule
from ase.units import Ha
from ase_dirac import DIRAC

As a first test we run an SCF calculation for the hydrogenchloride molecule and print the energy.

In [ ]:
molecule_label = 'HCl'
ase_molecule = molecule(molecule_label)
ase_molecule.calc = DIRAC(
              wave_function={'.scf': '',
                             '*scf': {'.ergcnv': '1.E-8 1.E-6',
                                      '.maxitr': '35'},
                            },
              molecule={'*basis': {'.default': 'cc-pVDZ'}},
              label=molecule_label)
print(f'The Hartree-Fock energy of {molecule_label} is ',ase_molecule.get_potential_energy() / Ha,' Hartree.')

In the next cell we run a more extensive calculation in which we tabulate MP2 energies computed with different Hamiltonians for three small molecules. In this piece of Python code we also allow for potential failures of a calculation, using the `try` and `except` mechanism of Python.

In [ ]:
# Some simple small molecules taken from the G2 set in the ASE database
molecules = ['H2O','N2','F2']
# Defines the **HAMILTONIAN input block, note the two lines of input for the full DCG Hamiltonian
hamiltonian_inputs = ['.nonrel','.x2cmmf','.gaunt\n.dossss']
# Labeling the different inputs with a descriptive string
calculation_labels = ['NR-MP2','X2Cmmf-MP2','DCG-MP2']
# Making an empty list to store results (energies) for each molecule
results = []

# Outermost loop over different molecules
for mol in molecules:
    ase_molecule = molecule(mol)
    energy = []
    # Loop over the three different Hamiltonians that we consider
    for ham, calc in zip(hamiltonian_inputs,calculation_labels):
        label=calc+'_'+mol
        try:
            ase_molecule.calc = DIRAC(
              hamiltonian={ham: ''},
              wave_function={'.scf': '', '.mp2': '',
                             '*scf': {'.ergcnv': '1.E-8 1.E-6',
                                      '.maxitr': '35'},
                             '*mp2cal': {'.occup': 'all','.virtual': 'all'},
                            },
              molecule={'*basis': {'.default': 'cc-pVDZ'}},
              label=label)
            energy.append(ase_molecule.get_potential_energy() / Ha)
        except:
            print (f'Calculation {label} could not be run correctly, please check')
            energy.append(0)
    results.append(energy)
    
# Print results in a table
print(' Møller-Plesset second order (MP2) energies computed with different Hamiltonians\n')
print(80*'-')
print('  {:<15} {:>20} {:>20} {:>20}'.format('Molecule',*calculation_labels))
print(80*'-')
for mol, result in zip(molecules,results):
    print ('  {:<15} {:>20.8f} {:>20.8f} {:>20.8f}'.format(mol,*result))
print(80*'-')

Note that the relativistic energies are always lower (more negative) than the non-relativistic ones, and the good approximation that the X2Cmmf method provides, with energies slightly below the DCG reference due to the neglect of the Gaunt interaction in the default X2Cmmf scheme.

Note that the current interface is rather minimal. We plan to add more functionality in next releases but of course also welcome user contributions (simply make a pull request if you have added something useful to this interface).

---

Added a small interface to be able to use DIRAC input files without the need of "translating" from DIRAC input to the dictionaries used by the ase_dirac.DIRAC calculator class:

In [ ]:
xyz_string = """2
F2
O      0 0 0
O      0.00000       -0.75545     -0.47116
"""

fscc_str = """ **DIRAC
    .TITLE
    Molecular Te. DEA from the dianion.
    .WAVE F
    **HAMILTONIAN
    .LVCORR
    **WAVE FUNCTIONS
    .SCF
    .RELCCSD
    ! because we want this
    *SCF
    .PRINT 
    1
    !.CLOSED SHELL
    !54 52
    **RELCCSD
    .ENERGY
    
    
    .PRINT
    1
    .FOCKSP
    *CCFSPC
    .DOIE2
    .NACTH
    1 1 0 0
    .MAXIT
    200
    **MOLECULE
    *BASIS
    .DEFAULT
    dyall.3zp
    ! **Moltra
    ! .Active
    ! energy -10.0 20.0 1.0
    *END OF
"""

with open("molecule.xyz", "w") as f:
    f.write(xyz_string)

with open("dirac_example.inp", "w") as f:
    f.write(fscc_str)

In [ ]:
ase_molecule = read("molecule.xyz", format="xyz")
ase_molecule.calc = DIRAC.from_file("dirac_example.inp")

# Also it can be called from string as:
# ase_molecule.calc = DIRAC.from_string(fscc_str)

param_list = [
    f"{key}, {ase_molecule.calc.parameters[key]}\n"
    for key in ase_molecule.calc.parameters.keys()
]

print("".join(param_list))

Where the default options in the `DIRAC.check_dirac_attributes()` method conflict with either inputted keywords or default `pam` parsing/default execution:
- `*charge: {'.charge': '0'}`: Occupation was established with `.CLOSED SHELL \n 54 52`. For the diatomic $O_2$ as proposed, the correct charge would be $-90$. This is not a big deal as occupation overrides charge. However this is redundant as the DIRAC parsing subroutine will default to $0$ in the absence of keyword.
- `*symmetry: {'.nosym': '#'}`: Linear symmetry was implicitly defined with `.CLOSED SHELL \n 54 52`. However, it is overriden, when `pam` would try to detect symmetry. This is a big deal as symmetry is ignored. 
- `integrals, {'.nucmod': '1'}` and `moltra, {'.active': 'all'}`: `moltra, {'.active': 'all'}` directly contradicts the default `MOLTRA` behaviour with `energy -10.0 20.0 1.0`. This can present issues in the correlation calculations, particularly in the `RELCCSD` module as it will be shown later, where the recalculation of the energy does not lead to the same values. 

In [ ]:
ase_molecule.get_potential_energy() / Ha

Also the multiplying the value by `Ha` when parsing to then later have to divide the energy by `Ha` to match the DIRAC output. 

```python
with open(output_filename, "r") as file:
        content = file.read()  # Read the entire content of the file

    ehfmatch = re.search(_scf_energy_re, content)
    if ehfmatch is not None:
        energy = float(ehfmatch.group(1)) * Ha
```

In [ ]:
!grep "Vi  10   24    1g  " dirac.out -A  23

Which is not ideal. However, we can still use the template from the calculation to impose:

In [ ]:
ase_molecule.calc.parameters.moltra.update({".active": "energy -10 20 1"})
ase_molecule.calc.parameters.molecule.update({"*symmetry": {".LINEAR": ""}})

param_list = [
    f"{key}, {ase_molecule.calc.parameters[key]}\n"
    for key in ase_molecule.calc.parameters.keys()
]

print("".join(param_list))

try:
    ase_molecule.calc.calculate()
except TypeError:
    ase_molecule.get_potential_energy() / Ha